# NanoDet-Plus Training for LEGO Chamber Detection

In [ ]:
!git clone https://github.com/RangiLyu/nanodet.git /kaggle/working/nanodet
%cd /kaggle/working/nanodet
!pip install -e . -q
!pip install onnx onnxsim onnxruntime pycocotools -q

In [ ]:
import json, os
from pathlib import Path

DATASET_DIR = Path("/kaggle/input/lego-chamber-detection")
WORK_DIR = Path("/kaggle/working")

# Verify dataset
for split in ["train", "val", "test"]:
    ann = json.load(open(DATASET_DIR / "annotations" / f"{split}.json"))
    print(f"{split}: {len(ann['images'])} images, {len(ann['annotations'])} annotations")

In [ ]:
import yaml, subprocess, time, json
from pathlib import Path

VARIANTS = [
    {"name": "nanodet-plus-m-320", "model_size": "1.0x", "imgsz": 320, "fpn_in": [116,232,464], "fpn_out": 96, "aux_in": 192, "aux_feat": 192, "batch": 96, "epochs": 200},
    {"name": "nanodet-plus-m-416", "model_size": "1.0x", "imgsz": 416, "fpn_in": [116,232,464], "fpn_out": 96, "aux_in": 192, "aux_feat": 192, "batch": 64, "epochs": 200},
    {"name": "nanodet-plus-m-1.5x-416", "model_size": "1.5x", "imgsz": 416, "fpn_in": [176,352,704], "fpn_out": 128, "aux_in": 256, "aux_feat": 256, "batch": 32, "epochs": 200},
]

DATASET_DIR = Path("/kaggle/input/lego-chamber-detection")

def generate_config(v, save_dir):
    return f"""
save_dir: {save_dir}
check_point_save_period: 10
keep_checkpoint_max: 3
log:
  interval: 50
model:
  weight_averager:
    name: ExpMovingAverager
    decay: 0.9998
  arch:
    name: NanoDetPlus
    detach_epoch: 10
    backbone:
      name: ShuffleNetV2
      model_size: {v['model_size']}
      out_stages: [2, 3, 4]
      activation: LeakyReLU
    fpn:
      name: GhostPAN
      in_channels: {v['fpn_in']}
      out_channels: {v['fpn_out']}
      kernel_size: 5
      num_extra_level: 1
      use_depthwise: True
      activation: LeakyReLU
    head:
      name: NanoDetPlusHead
      num_classes: 1
      input_channel: {v['fpn_out']}
      feat_channels: {v['fpn_out']}
      stacked_convs: 2
      kernel_size: 5
      strides: [8, 16, 32, 64]
      activation: LeakyReLU
      reg_max: 7
      norm_cfg:
        type: BN
      loss:
        loss_qfl:
          name: QualityFocalLoss
          use_sigmoid: True
          beta: 2.0
          loss_weight: 1.0
        loss_dfl:
          name: DistributionFocalLoss
          loss_weight: 0.25
        loss_bbox:
          name: GIoULoss
          loss_weight: 2.0
    aux_head:
      name: SimpleConvHead
      num_classes: 1
      input_channel: {v['aux_in']}
      feat_channels: {v['aux_feat']}
      stacked_convs: 4
      strides: [8, 16, 32, 64]
      activation: LeakyReLU
      reg_max: 7
data:
  train:
    name: CocoDataset
    img_path: {DATASET_DIR}/images/train
    ann_path: {DATASET_DIR}/annotations/train.json
    input_size: [{v['imgsz']}, {v['imgsz']}]
    keep_ratio: False
    pipeline:
      perspective: 0.0
      scale: [0.6, 1.4]
      stretch: [[0.8, 1.2], [0.8, 1.2]]
      rotation: 0
      shear: 0
      translate: 0.2
      flip: 0.5
      brightness: 0.2
      contrast: [0.6, 1.4]
      saturation: [0.5, 1.2]
      normalize: [[103.53, 116.28, 123.675], [57.375, 57.12, 58.395]]
  val:
    name: CocoDataset
    img_path: {DATASET_DIR}/images/val
    ann_path: {DATASET_DIR}/annotations/val.json
    input_size: [{v['imgsz']}, {v['imgsz']}]
    keep_ratio: False
    pipeline:
      normalize: [[103.53, 116.28, 123.675], [57.375, 57.12, 58.395]]
device:
  gpu_ids: [0]
  workers_per_gpu: 4
  batchsize_per_gpu: {v['batch']}
  precision: 32
schedule:
  optimizer:
    name: AdamW
    lr: 0.001
    weight_decay: 0.05
  warmup:
    name: linear
    steps: 500
    ratio: 0.0001
  total_epochs: {v['epochs']}
  lr_schedule:
    name: CosineAnnealingLR
    T_max: {v['epochs']}
    eta_min: 0.0
  val_intervals: 10
grad_clip: 35
evaluator:
  name: CocoDetectionEvaluator
  save_key: mAP
class_names: ['piece']
"""

results = {}
for v in VARIANTS:
    print(f"\n{'='*60}")
    print(f"Training {v['name']} ({v['imgsz']}x{v['imgsz']})")
    print(f"{'='*60}")

    save_dir = f"/kaggle/working/runs/{v['name']}"
    config_path = f"/kaggle/working/configs/{v['name']}.yml"
    os.makedirs(os.path.dirname(config_path), exist_ok=True)

    with open(config_path, "w") as f:
        f.write(generate_config(v, save_dir))

    start = time.time()
    result = subprocess.run(
        ["python", "tools/train.py", config_path],
        cwd="/kaggle/working/nanodet",
    )
    elapsed = time.time() - start

    results[v['name']] = {
        "returncode": result.returncode,
        "elapsed_min": round(elapsed / 60, 1),
    }
    print(f"\n{v['name']} finished in {elapsed/60:.1f} min (rc={result.returncode})")

In [ ]:
import subprocess, glob

for v in VARIANTS:
    save_dir = f"/kaggle/working/runs/{v['name']}"
    best_pth = glob.glob(f"{save_dir}/**/nanodet_model_best.pth", recursive=True)
    if not best_pth:
        print(f"No checkpoint found for {v['name']}")
        continue

    config_path = f"/kaggle/working/configs/{v['name']}.yml"
    onnx_path = f"/kaggle/working/exports/{v['name']}.onnx"
    os.makedirs("/kaggle/working/exports", exist_ok=True)

    subprocess.run([
        "python", "tools/export_onnx.py",
        "--cfg_path", config_path,
        "--model_path", best_pth[0],
        "--out_path", onnx_path,
    ], cwd="/kaggle/working/nanodet")

    # Simplify
    sim_path = onnx_path.replace(".onnx", "-sim.onnx")
    subprocess.run(["python", "-m", "onnxsim", onnx_path, sim_path])

    if os.path.exists(sim_path):
        size_kb = os.path.getsize(sim_path) / 1024
        print(f"{v['name']}: exported {size_kb:.0f} KB")
    results[v['name']]["onnx_size_kb"] = size_kb if os.path.exists(sim_path) else None

In [ ]:
import json, glob, os, numpy as np
import onnxruntime as ort
import cv2

DATASET_DIR = Path("/kaggle/input/lego-chamber-detection")

def evaluate_onnx(onnx_path, test_ann_path, test_img_dir, imgsz, conf=0.25):
    """Quick evaluation: count accuracy on test set."""
    session = ort.InferenceSession(str(onnx_path))
    input_name = session.get_inputs()[0].name

    ann = json.load(open(test_ann_path))
    # Group annotations by image
    img_anns = {}
    for a in ann["annotations"]:
        img_anns.setdefault(a["image_id"], []).append(a)

    mean = np.array([103.53, 116.28, 123.675], dtype=np.float32)
    std = np.array([57.375, 57.12, 58.395], dtype=np.float32)

    correct, total = 0, 0
    latencies = []

    for img_info in ann["images"]:
        img_path = os.path.join(test_img_dir, img_info["file_name"])
        img = cv2.imread(img_path)
        if img is None: continue

        gt_count = len(img_anns.get(img_info["id"], []))

        # Preprocess
        resized = cv2.resize(img, (imgsz, imgsz)).astype(np.float32)
        blob = np.transpose((resized - mean) / std, (2,0,1))[None].astype(np.float32)

        import time
        t0 = time.perf_counter()
        outputs = session.run(None, {input_name: blob})
        latencies.append((time.perf_counter() - t0) * 1000)

        # Decode (simplified - count predictions above threshold)
        preds = outputs[0][0]
        num_reg = 4 * 8  # reg_max=7, 4*(7+1)=32
        cls_scores = 1.0 / (1.0 + np.exp(-preds[:, :1]))  # 1 class
        pred_count = int((cls_scores.max(axis=1) >= conf).sum())

        # Decision match
        gt_dec = "empty" if gt_count == 0 else "single" if gt_count == 1 else "multi"
        pred_dec = "empty" if pred_count == 0 else "single" if pred_count == 1 else "multi"
        if gt_dec == pred_dec: correct += 1
        total += 1

    return {
        "decision_match_rate": correct / total if total else 0,
        "samples": total,
        "avg_latency_ms": np.mean(latencies) if latencies else 0,
    }

final_results = {}
for v in VARIANTS:
    onnx_path = f"/kaggle/working/exports/{v['name']}-sim.onnx"
    if not os.path.exists(onnx_path):
        onnx_path = f"/kaggle/working/exports/{v['name']}.onnx"
    if not os.path.exists(onnx_path):
        print(f"No ONNX for {v['name']}")
        continue

    metrics = evaluate_onnx(
        onnx_path,
        str(DATASET_DIR / "annotations" / "test.json"),
        str(DATASET_DIR / "images" / "test"),
        v["imgsz"],
    )
    metrics["model"] = v["name"]
    metrics["onnx_size_kb"] = os.path.getsize(onnx_path) / 1024
    final_results[v["name"]] = metrics
    print(f"{v['name']}: decision={metrics['decision_match_rate']:.1%}, latency={metrics['avg_latency_ms']:.1f}ms, size={metrics['onnx_size_kb']:.0f}KB")

# Save all results
with open("/kaggle/working/nanodet_results.json", "w") as f:
    json.dump(final_results, f, indent=2)
print("\nResults saved to /kaggle/working/nanodet_results.json")

In [ ]:
import glob
!mkdir -p /kaggle/working/output
!cp /kaggle/working/exports/*-sim.onnx /kaggle/working/output/ 2>/dev/null || true
!cp /kaggle/working/exports/*.onnx /kaggle/working/output/ 2>/dev/null || true
!cp /kaggle/working/nanodet_results.json /kaggle/working/output/
# Copy best checkpoints
for v in VARIANTS:
    ckpts = glob.glob(f"/kaggle/working/runs/{v['name']}/**/nanodet_model_best.pth", recursive=True)
    if ckpts:
        import shutil
        shutil.copy2(ckpts[0], f"/kaggle/working/output/{v['name']}_best.pth")

!ls -la /kaggle/working/output/
print("Done! Download output files from the Output tab.")